In [15]:
import tensorflow as tf
from pathlib import Path

print("TF:", tf.__version__)          # note this — goes into requirements.txt later
print("GPU:", tf.config.list_physical_devices("GPU"))

# find repo root regardless of where the kernel starts
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR  = ROOT / "Downloads" / "ai-deepfakedetection-main" / "backend" / "train" / "dataset"         # dataset
MODEL_DIR = ROOT / "Downloads" / "ai-deepfakedetection-main" / "backend" /"saved_models"  # output
print("Repo root:", ROOT)

TF: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Repo root: /Users/priyankareddy


In [16]:
import zipfile, os

if not (DATA_DIR / "train").exists():
    !kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
    with zipfile.ZipFile("cifake-real-and-ai-generated-synthetic-images.zip") as z:
        z.extractall(DATA_DIR)

print(os.listdir(DATA_DIR))    # expect: ['train', 'test']

['.DS_Store', 'test', 'train']


In [17]:
IMG_SIZE = 32
BATCH = 256

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "train"), validation_split=0.1, subset="training", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="binary")

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "train"), validation_split=0.1, subset="validation", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="binary")

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "test"), image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
    label_mode="binary", shuffle=False)

train_ds = train_ds_raw.cache().prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds_raw.cache().prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds_raw.prefetch(tf.data.AUTOTUNE)

print(train_ds_raw.class_names)   # ['FAKE', 'REAL'] → 0 = FAKE, 1 = REAL

Found 200000 files belonging to 2 classes.
Using 180000 files for training.
Found 200000 files belonging to 2 classes.
Using 20000 files for validation.
Found 30000 files belonging to 2 classes.
['fake', 'real']


In [18]:
from tensorflow.keras import layers

base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
])

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = data_aug(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)   # P(REAL)

model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1, 1, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [19]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

history1 = model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5


/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


704/704 ━━━━━━━━━━━━━━━━━━━━ 87s 118ms/step - accuracy: 0.6431 - auc: 0.7019 - loss: 0.6260 - val_accuracy: 0.6941 - val_auc: 0.7729 - val_loss: 0.5789
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 70s 99ms/step - accuracy: 0.6660 - auc: 0.7329 - loss: 0.6030 - val_accuracy: 0.7039 - val_auc: 0.7861 - val_loss: 0.5667
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 79s 112ms/step - accuracy: 0.6667 - auc: 0.7333 - loss: 0.6025 - val_accuracy: 0.7082 - val_auc: 0.7918 - val_loss: 0.5625
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 84s 119ms/step - accuracy: 0.6704 - auc: 0.7365 - loss: 0.5999 - val_accuracy: 0.7104 - val_auc: 0.7934 - val_loss: 0.5599
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 76s 108ms/step - accuracy: 0.6692 - auc: 0.7364 - loss: 0.6000 - val_accuracy: 0.7106 - val_auc: 0.7945 - val_loss: 0.5582


In [20]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=5,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True)])

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 86s 116ms/step - accuracy: 0.6869 - auc: 0.7603 - loss: 0.5789 - val_accuracy: 0.7285 - val_auc: 0.8194 - val_loss: 0.5264
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 62s 87ms/step - accuracy: 0.7046 - auc: 0.7829 - loss: 0.5560 - val_accuracy: 0.7380 - val_auc: 0.8290 - val_loss: 0.5112
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 61s 87ms/step - accuracy: 0.7122 - auc: 0.7938 - loss: 0.5442 - val_accuracy: 0.7440 - val_auc: 0.8364 - val_loss: 0.5035
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 60s 85ms/step - accuracy: 0.7192 - auc: 0.8027 - loss: 0.5341 - val_accuracy: 0.7521 - val_auc: 0.8411 - val_loss: 0.4937
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 59s 84ms/step - accuracy: 0.7251 - auc: 0.8090 - loss: 0.5272 - val_accuracy: 0.7534 - val_auc: 0.8447 - val_loss: 0.4904


In [21]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

loss, acc, auc = model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f} | AUC: {auc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds]).ravel()
y_pred = (model.predict(test_ds).ravel() > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"]))
print(confusion_matrix(y_true, y_pred))

118/118 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.7116 - auc: 0.7911 - loss: 0.5602
Test accuracy: 0.7116 | AUC: 0.7911


2026-09-19 19:07:45.253157: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


118/118 ━━━━━━━━━━━━━━━━━━━━ 11s 79ms/step
              precision    recall  f1-score   support

        FAKE       0.55      0.75      0.63     10000
        REAL       0.85      0.69      0.76     20000

    accuracy                           0.71     30000
   macro avg       0.70      0.72      0.70     30000
weighted avg       0.75      0.71      0.72     30000

[[ 7510  2490]
 [ 6162 13838]]


In [22]:
import json

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / "model.keras")

meta = {
    "model": "EfficientNetB0",
    "input_size": [IMG_SIZE, IMG_SIZE],
    "labels": {"0": "FAKE", "1": "REAL"},
    "preprocessing": "RGB, resize to 32x32, keep values 0-255 (no rescaling)",
    "output": "sigmoid, P(REAL)",
    "test_accuracy": float(acc),
}
with open(MODEL_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", list(MODEL_DIR.iterdir()))

Saved: [PosixPath('/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/saved_models/model.keras'), PosixPath('/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/saved_models/meta.json')]
